# 在 Bedrock AgentCore Runtime 中部署 Strands Agent 并实现可观测性

本动手实验演示如何使用 Amazon Bedrock AgentCore Runtime 部署 Strands Agents，集成内置工具和自定义函数，实现全面的 AI 代理能力。

## 概述

在本实验中，您将：
- 将带有工具的 Strands Agents 部署到 Bedrock AgentCore Runtime
- 使用 `boto3` 通过 IAM 权限调用已部署的代理
- 了解 Bedrock AgentCore Runtime 会话的特性
- 学习 GenAI 可观测性和可追溯性

## 前提条件

在开始本实验之前，请确保您已具备：
- 已配置 AWS 凭证（IAM 角色或环境变量）
- 已安装所需的 Python 包
- 基于 AWS 区域的 Nova Pro 模型 ID
- 已为 Bedrock AgentCore 可观测性启用 CloudWatch Transaction Search

如果您未在已承担 IAM 角色的环境中运行，请将 AWS 凭证设置为环境变量：

In [ ]:
import os

#os.environ["AWS_ACCESS_KEY_ID"]=<YOUR ACCESS KEY>
#os.environ["AWS_SECRET_ACCESS_KEY"]=<YOUR SECRET KEY>
#os.environ["AWS_SESSION_TOKEN"]=<OPTIONAL - YOUR SESSION TOKEN IF TEMP CREDENTIAL>
#os.environ["AWS_REGION"]=<AWS REGION WITH BEDROCK AGENTCORE AVAILABLE>

安装 Strands Agents 和 Bedrock AgentCore SDK 所需的包：

In [ ]:
#%pip install -q strands-agents strands-agents-tools bedrock-agentcore bedrock-agentcore-starter-toolkit

根据 AWS 区域设置 Nova Pro 模型 ID：

In [ ]:
import boto3

region = boto3.session.Session().region_name

NOVA_PRO_MODEL_ID = "us.amazon.nova-pro-v1:0"
if region.startswith("eu"):
    NOVA_PRO_MODEL_ID = "eu.amazon.nova-pro-v1:0"
elif region.startswith("ap"):
    NOVA_PRO_MODEL_ID = "apac.amazon.nova-pro-v1:0"

print(f"Nova Pro Model ID: {NOVA_PRO_MODEL_ID}")

启用 [**CloudWatch APM → Transaction Search**](https://console.aws.amazon.com/cloudwatch/home#logsV2:transaction-search) 以实现对已部署 Strands Agent 的全面可观测性和监控。在本实验中，将 **X-Ray trace indexing 设置为 100%**，以生成所有追踪摘要，用于端到端事务分析。

![bedrock-agentcore-observability-setup](images/observability-setup.png)

## 什么是 Strands Agent 与 Bedrock AgentCore Runtime？

Strands Agents 提供了一个强大的框架，用于构建具有内置和自定义工具集成的 AI 代理。当与 Bedrock AgentCore Runtime 一起部署时，您将获得：

- **可扩展部署**：具有自动扩展能力的托管基础设施
- **安全认证**：内置 Cognito 集成的安全机制
- **可观测性**：集成 CloudWatch 进行监控和调试
- **工具集成**：将计算器等内置工具与自定义函数相结合

In [ ]:
from strands import Agent, tool
from strands.models import BedrockModel
from strands_tools import calculator

# 创建一个用于演示的自定义天气工具
@tool
def weather(city: str) -> str:
    """获取城市的天气信息
    Args:
        city: 城市或地点名称
    """
    return f"{city}的天气：晴天，35°C"  # 用于演示的虚拟结果

# Create and test the comprehensive Strands Agent
agent = Agent(
    model=BedrockModel(model_id=NOVA_PRO_MODEL_ID),
    system_prompt = """你是一个生活助手，运用科学的知识回答各种问题。""",
    tools=[weather, calculator],
)

agent("香港的天气怎么样？请用华氏度返回温度。")

## 将 Strands Agent 部署到 Bedrock AgentCore Runtime

创建 Strands Agent 的可部署版本，并将其部署为托管服务。
![bedrock-agentcore-runtime-launch](images/runtime-launch.png)

### 部署流程：
1. 创建包含代理配置的 Python 文件
2. 设置包含依赖项的 requirements.txt
3. 配置带有身份验证的 AgentCore Runtime
4. 使用 CodeBuild 进行容器化部署

### 步骤 1：创建包含代理配置的 Python 文件

创建定义 Strands Agent 的主 Python 文件，包含内置工具和自定义工具。该文件将作为容器化服务部署到 Bedrock AgentCore Runtime。

**部署要求：**
要将代理部署到 AgentCore Runtime，我们需要：
- 导入 Runtime App：`from bedrock_agentcore.runtime import BedrockAgentCoreApp`
- 初始化 App：`app = BedrockAgentCoreApp()`
- 使用 `@app.entrypoint` 装饰器标记调用函数
- 让 AgentCore Runtime 通过 `app.run()` 控制执行

In [ ]:
%%writefile strands_agent.py
from strands import Agent, tool
from strands.models import BedrockModel
from strands_tools import calculator
import boto3
from bedrock_agentcore.runtime import BedrockAgentCoreApp

app = BedrockAgentCoreApp()

# Setup Nova Pro model ID based on AWS region
NOVA_PRO_MODEL_ID = "us.amazon.nova-pro-v1:0"
region = boto3.session.Session().region_name
if region.startswith("eu"):
    NOVA_PRO_MODEL_ID = "eu.amazon.nova-pro-v1:0"
elif region.startswith("ap"):
    NOVA_PRO_MODEL_ID = "apac.amazon.nova-pro-v1:0"

# 创建一个用于演示的自定义天气工具
@tool
def weather(city: str) -> str:
    """获取城市的天气信息
    Args:
        city: 城市或地点名称
    """
    return f"{city}的天气：晴天，35°C"  # 用于演示的虚拟结果


# Create and test the comprehensive Strands Agent
agent = Agent(
    model=BedrockModel(model_id=NOVA_PRO_MODEL_ID),
    system_prompt = """你是一个生活助手，运用科学的知识回答各种问题。""",
    tools=[weather, calculator],
)

@app.entrypoint
async def strands_agent_bedrock(payload, context):
    """
    Invoke the agent with a payload
    """
    print(f"Payload: {payload}")
    print(f"Context: {context}")
    user_input = payload.get("prompt", "No prompt found")
    response = agent(user_input)
    return response
    
    # Streaming Mode
    """
    stream = agent.stream_async(user_input)
    async for event in stream:
        if "data" in event:
            yield event
    """

if __name__ == "__main__":
    app.run()

## 幕后发生了什么？

当您使用 `BedrockAgentCoreApp` 时，它会自动：

* 创建一个监听 8080 端口的 HTTP 服务器
* 实现处理代理请求所需的 `/invocations` 端点
* 实现用于健康检查的 `/ping` 端点（对异步代理非常重要）
* 处理正确的内容类型和响应格式
* 按照 AWS 标准进行错误处理

### 本地测试（可选 - 如果动手实验环境中 8080 端口被占用，请跳过）

在部署到 AgentCore Runtime 之前，您可以在本地测试代理以验证功能。

**启动本地服务器：**
```bash
cd 05-bedrock-agentcore-runtime-and-observability/
uv run strands_agent.py
```
或
```bash
cd 05-bedrock-agentcore-runtime-and-observability/
python strands_agent.py
```
服务器将在 `http://localhost:8080` 启动

**在另一个终端中使用 cURL 测试：**
```bash
curl -X POST http://localhost:8080/invocations \
  -H "Content-Type: application/json" \
  -H "X-Amzn-Bedrock-AgentCore-Runtime-User-Id: 123" \
  -H "X-Amzn-Bedrock-AgentCore-Runtime-Session-Id: 1234567890123456789012345678901234567890" \
  -d '{"prompt": "How is the weather in HK?"}'
```

**停止本地服务器：**

在运行 `strands_agent.py` 的终端中按 `Ctrl+C` 停止本地运行的 AgentCore Runtime。

### 步骤 2：创建依赖文件

定义 Strands Agent 部署所需的 Python 依赖项。

**关键依赖项：**
- **aws-opentelemetry-distro**：AgentCore 可观测性所需
- **strands-agents**：Strands 核心框架
- **bedrock-agentcore**：Runtime 集成

**可观测性集成：**
`aws-opentelemetry-distro` 库支持自动检测，用于监控和追踪。如 [AgentCore 可观测性指南](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/observability-configure.html) 中所述，容器化环境（如 docker）需要添加以下命令：

```dockerfile
CMD ["opentelemetry-instrument", "python", "main.py"]
```

这种自动检测方法会自动将 OpenTelemetry SDK 添加到 Python 路径中，以实现全面的可观测性。

In [ ]:
%%writefile requirements.txt
strands-agents
strands-agents-tools
bedrock-agentcore
bedrock-agentcore-starter-toolkit
boto3
aws-opentelemetry-distro>=0.10.0

### 步骤 3：配置 AgentCore Runtime

设置 Bedrock AgentCore Runtime 配置，支持自动创建资源。

**生成的构件：**
此步骤会创建必要的部署文件：
- **Dockerfile**：Strands Agent 的容器配置
- **.dockerignore**：列出 docker 构建时排除的文件
- **.bedrock_agentcore.yaml**：Runtime 部署配置

请注意，当使用 bedrock_agentcore_starter_toolkit 配置代理时，它会自动处理 opentelemetry 检测。生成的 Dockerfile 将包含：
```bash
CMD ["opentelemetry-instrument", "python", "runtime_agent_main.py"]
```

In [ ]:
from bedrock_agentcore_starter_toolkit import Runtime
import boto3

region = boto3.session.Session().region_name
agentcore_runtime = Runtime()

response = agentcore_runtime.configure(
    entrypoint="strands_agent.py",
    auto_create_execution_role=True,
    auto_create_ecr=True,
    requirements_file="requirements.txt",
    region=region,
    agent_name="strands_getting_started"
)
response

### 步骤 4：部署到 AgentCore Runtime

使用 AWS CodeBuild 启动容器化和部署流程。

**部署流程：**
- 构建 Strands Agents 的容器化版本
- 创建所需的 AWS 资源（ECR 仓库、IAM 角色）
- 将容器镜像推送到 Amazon ECR
- 部署到 AgentCore Runtime 作为托管的自动扩展服务

In [ ]:
launch_result = agentcore_runtime.launch()

### 步骤 5：验证部署状态

监控部署进度并等待 Runtime 就绪：

In [ ]:
import time

print("Checking AgentCore Runtime status...")
status_response = agentcore_runtime.status()
status = status_response.endpoint['status']
print(f"Initial status: {status}")

end_status = ['READY', 'CREATE_FAILED', 'DELETE_FAILED', 'UPDATE_FAILED']
while status not in end_status:
    print(f"Status: {status} - waiting...")
    time.sleep(10)
    status_response = agentcore_runtime.status()
    status = status_response.endpoint['status']

if status == 'READY':
    print("✓ AgentCore Runtime is READY!")
else:
    print(f"⚠ AgentCore Runtime status: {status}")
    
print(f"Final status: {status}")

agent_runtime_id = launch_result.agent_id
agent_runtime_arn = launch_result.agent_arn
ecr_repo_name = launch_result.ecr_uri.split('/')[1]
codebuild_name = launch_result.codebuild_id.split(':')[0]
print(f"Strands AgentCore Runtime ID: {agent_runtime_id}")
print(f"Strands AgentCore Runtime ARN: {agent_runtime_arn}")
print(f"ECR Repo for Strands AgentCore Runtime: {ecr_repo_name}")
print(f"CodeBuild Project for Strands AgentCore Runtime: {codebuild_name}")

## 测试已部署的代理

通过 Bedrock AgentCore Runtime API 使用各种提示调用已部署的 Strands Agent，以验证所有工具是否正常工作。

In [ ]:
import boto3
import json
import uuid

SESSION_ID = str(uuid.uuid4())

PROMPT = "香港的天气怎么样？请用华氏度返回温度。"

agentcore_client = boto3.client(
    'bedrock-agentcore',
    region_name=boto3.session.Session().region_name
)

boto3_response = agentcore_client.invoke_agent_runtime(
    agentRuntimeArn=agent_runtime_arn,
    qualifier="DEFAULT",
    runtimeSessionId=SESSION_ID, #Provide same session identifier across multiple requests to maintain conversation context, and with better traceability
    payload=json.dumps({"prompt": PROMPT})
)

if "text/event-stream" in boto3_response.get("contentType", ""):
    for line in boto3_response["response"].iter_lines(chunk_size=1):
        if line:
            line = line.decode("utf-8")
            if line.startswith("data: "):
                print(line)
else:
    events = []
    for event in boto3_response.get("response", []):
        print(event.decode('utf-8'))
        events.append(event)

### 测试对话历史

验证代理是否能够使用相同的会话 ID 在多次请求之间保持对话上下文：

In [ ]:
PROMPT = "我们之前聊了什么？"

boto3_response = agentcore_client.invoke_agent_runtime(
    agentRuntimeArn=agent_runtime_arn,
    qualifier="DEFAULT",
    runtimeSessionId=SESSION_ID, #Provide same session identifier across multiple requests to maintain conversation context, and with better traceability
    payload=json.dumps({"prompt": PROMPT})
)

if "text/event-stream" in boto3_response.get("contentType", ""):
    for line in boto3_response["response"].iter_lines(chunk_size=1):
        if line:
            line = line.decode("utf-8")
            if line.startswith("data: "):
                print(line)
else:
    events = []
    for event in boto3_response.get("response", []):
        print(event.decode('utf-8'))
        events.append(event)

## 对话历史和会话管理

**Strands Agents 对话管理：**
Strands Agents 包含内置的对话管理功能，默认使用 `SlidingWindowConversationManager` 策略。这会自动维护会话内的对话上下文，但不会在多个会话之间持久化。

**参考：** [Strands Agents Conversation Management](https://strandsagents.com/latest/documentation/docs/user-guide/concepts/agents/conversation-management/)

**AgentCore Runtime 会话隔离：**
- **会话标识**：通过应用程序提供的唯一 `runtimeSessionId` 进行标识，如果 `runtimeSessionId` 留空，则由 Runtime 在首次调用时自动生成
- **会话隔离**：在专用 microVM 中运行，具有完全隔离的 CPU、内存和文件系统资源
- **上下文保留**：在同一对话中的多次交互之间保留上下文
- **会话持续时间**：会话最长持续 8 小时，或在 15 分钟不活动后结束
- **自动清理**：会话终止后，整个 microVM 将被终止并清理内存

**参考：** [AgentCore Runtime Sessions](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/runtime-sessions.html)


如需超出会话持续时间的持久化记忆，请集成 **Bedrock AgentCore Memory** 以实现长期对话存储和检索。

**参考：** [AgentCore Memory: Add memory to your AI agent](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/memory.html)

## Amazon CloudWatch 上的 AgentCore 可观测性

### 什么是 Bedrock AgentCore 可观测性？

Amazon Bedrock AgentCore 通过 CloudWatch 和 X-Ray 集成提供内置的可观测性。这使得监控代理性能、追踪请求流程和分析对话模式成为可能。

主要功能：
- **会话追踪**：监控单个对话
- **分布式追踪**：跨组件追踪请求
- **性能指标**：延迟、吞吐量和错误率
- **Span 分析**：详细的执行分解

了解更多：[AgentCore Observability Guide](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/observability.html)

### 查看主仪表板

等待 2-3 分钟，然后访问 [**Amazon CloudWatch 控制台**](https://console.aws.amazon.com/cloudwatch/home#/gen-ai-observability/agent-core) 查看您的 AgentCore 可观测性仪表板：
![observability-main-dashboard.png](images/observability-main-dashboard.png)

### 会话管理

点击 `strands_getting_started` 中的 **DEFAULT** 查看会话历史。这将显示我们的测试会话，其中包含两条追踪记录："How is the weather in HK?" 和 "What did I ask?"
![observability-session-list.png](images/observability-session-list.png)

### 会话概览

选择一个会话以查看指标、追踪时间线和性能数据：
![observability-trace-list.png](images/observability-trace-list.png)

### 追踪分析

点击任意追踪记录以查看详细的执行步骤：

**Span 详情**：
![observability-trace-span-1.png](images/observability-trace-span-1.png)
![observability-trace-span-2.png](images/observability-trace-span-2.png)
![observability-trace-span-3.png](images/observability-trace-span-3.png)

## 资源清理（可选）

清理已部署的资源：

In [ ]:
import boto3
import os

agentcore_control_client = boto3.client('bedrock-agentcore-control', region_name=region)
ecr_client = boto3.client('ecr',region_name=region)
codebuild_client = boto3.client('codebuild',region_name=region)

try:
    print("Deleting AgentCore Runtime...")
    agentcore_control_client.delete_agent_runtime(agentRuntimeId=agent_runtime_id)
    print("✓ AgentCore Runtime deletion initiated")

    print("Deleting ECR repository...")
    ecr_client.delete_repository(repositoryName=ecr_repo_name, force=True)
    print("✓ ECR repository deleted")

    print("Deleting CodeBuild Project...")
    codebuild_client.delete_project(name=codebuild_name)
    print("✓ CodeBuild Project deleted")

    print("Deleting Bedrock AgentCore configuration file...")
    os.remove(".bedrock_agentcore.yaml") 
    print("✓ .bedrock_agentcore.yaml deleted")
except Exception as e:
    print(f"❌ Error during cleanup: {e}")
    print("You may need to manually clean up some resources.")

## 总结

在本实验中，您成功完成了：

- ✅ 将 Strands Agent 部署到 Bedrock AgentCore Runtime
- ✅ 配置了 Amazon CloudWatch 的可观测性和监控
- ✅ 通过在 AgentCore Runtime 环境中调用远程代理，测试了端到端的代理功能
- ✅ 探索了会话隔离和自动清理能力

## AgentCore Runtime 对 Strands Agents 的主要优势

- **生产级部署**：为 AI 代理提供可扩展的托管基础设施
- **全面的可观测性**：内置监控、追踪和调试能力
- **会话管理**：自动会话隔离和清理
- **企业就绪**：为生产工作负载提供安全性、可靠性和合规性功能
